<a href="https://colab.research.google.com/github/yashwini321/yashwini/blob/main/spam.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [36]:
import pandas as pd
import pickle

from sklearn.model_selection import train_test_split

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D
from tensorflow.keras.layers import GlobalMaxPooling1D, Dense, Dropout

# Load email dataset
data = pd.read_csv("/content/spam.csv")

X = data["text"].astype(str)
y = (data["label"] == "spam").astype(int)

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Tokenization
tokenizer = Tokenizer(num_words=10000, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train = pad_sequences(
    tokenizer.texts_to_sequences(X_train),
    maxlen=100
)

X_test = pad_sequences(
    tokenizer.texts_to_sequences(X_test),
    maxlen=100
)

# CNN model
model = Sequential([
    Embedding(10000, 128, input_length=100),

    Conv1D(
        filters=128,
        kernel_size=5,
        activation="relu"
    ),

    GlobalMaxPooling1D(),

    Dropout(0.5),

    Dense(64, activation="relu"),

    Dropout(0.5),

    Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

# Train
history = model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=32,
    validation_split=0.2
)

# Test
loss, accuracy = model.evaluate(X_test, y_test)

print("Test Accuracy:", accuracy)

# Save model
model.save("spam_cnn.keras")

# Save tokenizer
with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

print("Model saved successfully!")

UnicodeDecodeError: 'utf-8' codec can't decode bytes in position 606-607: invalid continuation byte